# WordNet Cross-Attention Explore

This notebook follows the cache-first HotpotQA scan pattern used in `wordnet_embeds_explore.ipynb`, reuses the same phrase/token embedding index when it already exists, and adds a training-free DeBERTa sense-matching prototype.

Core idea:

- scan HotpotQA documents and build or reuse a token database
- mark a target token inside a context sentence
- encode `(context, sense profile)` with `microsoft/deberta-v3-large`
- pool target, local context, and definition/profile vectors
- add a cross-attention-like token alignment score
- rank candidate definitions without any training


In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import math
import pickle
import re
from typing import Any, Optional

import numpy as np
import spacy
import torch
import torch.nn.functional as F
from spacy.tokenizer import Tokenizer
from spacy.util import compile_infix_regex
from transformers import AutoModel, AutoTokenizer

from text_processing import (
    clean_text,
    encode_chunk_batch,
    extract_important_spans,
    get_token_indices_for_phrase,
    normalize_text,
)
from utils import build_hotpot_retrieval_dataset
from wordnet_explorer import WordNetExplorer

torch.set_grad_enabled(False)


In [ ]:
file_path = "hotpot_dev_distractor_v1.json"
num_samples = 500

spacy_model_name = "en_core_web_lg"
text_encoder_name_or_path = "microsoft/deberta-v3-large"
local_files_only = False

device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 8
remove_duplicate_token = True
discard_no_word = False
wordnet_candidate_limit = 8
max_length = 512
window_size = 3

output_path = Path("hotpot_phrase_token_embedding_index.pkl")
default_weights = {
    "cos_target_def": 0.35,
    "cos_window_def": 0.20,
    "cross_alignment": 0.35,
    "lexical_overlap": 0.10,
}

print(f"device={device}")
print(f"token db cache={output_path.resolve()}")


In [ ]:
nlp = None
tokenizer = None
text_encoder = None
wordnet_explorer = None


def load_custom_nlp(model_name: str):
    loaded_nlp = spacy.load(model_name)
    infixes = [pattern for pattern in loaded_nlp.Defaults.infixes if "-" not in pattern]
    infix_re = compile_infix_regex(infixes)
    loaded_nlp.tokenizer = Tokenizer(
        loaded_nlp.vocab,
        prefix_search=loaded_nlp.tokenizer.prefix_search,
        suffix_search=loaded_nlp.tokenizer.suffix_search,
        infix_finditer=infix_re.finditer,
        token_match=loaded_nlp.tokenizer.token_match,
    )
    return loaded_nlp


def load_text_encoder(name_or_path: str, device: str, local_files_only: bool = False):
    tokenizer_kwargs = {
        "use_fast": True,
        "local_files_only": local_files_only,
    }
    model_kwargs = {
        "local_files_only": local_files_only,
    }

    loaded_tokenizer = AutoTokenizer.from_pretrained(name_or_path, **tokenizer_kwargs)
    if not getattr(loaded_tokenizer, "is_fast", False):
        raise TypeError("A fast tokenizer is required because the prototype relies on offset mapping.")

    loaded_text_encoder = AutoModel.from_pretrained(name_or_path, **model_kwargs)
    loaded_text_encoder.to(device)
    loaded_text_encoder.eval()
    return loaded_tokenizer, loaded_text_encoder


def ensure_nlp_loaded():
    global nlp
    if nlp is None:
        nlp = load_custom_nlp(spacy_model_name)
        print(f"Loaded spaCy model: {spacy_model_name}")
    return nlp


def ensure_text_encoder_loaded():
    global tokenizer, text_encoder
    if tokenizer is None or text_encoder is None:
        tokenizer, text_encoder = load_text_encoder(
            text_encoder_name_or_path,
            device=device,
            local_files_only=local_files_only,
        )
        print(f"Loaded text encoder: {text_encoder_name_or_path} on {device}")
    return tokenizer, text_encoder


def ensure_wordnet_explorer_loaded():
    global wordnet_explorer
    if wordnet_explorer is None:
        wordnet_explorer = WordNetExplorer()
    return wordnet_explorer


In [ ]:
def collect_span_embedding_records(token_embeddings, offsets, span_list, kind, document_idx, title, text):
    records = []
    for term, start_char, end_char in span_list:
        token_indices = get_token_indices_for_phrase(start_char, end_char, offsets)
        if not token_indices:
            continue

        embedding = token_embeddings[token_indices].mean(dim=0).to(torch.float32).numpy()
        records.append(
            {
                "kind": kind,
                "term": term,
                "lookup_term": normalize_text(term),
                "document_idx": int(document_idx),
                "title": title,
                "span": (int(start_char), int(end_char)),
                "text": text,
                "embedding": embedding,
            }
        )
    return records


def build_phrase_token_embedding_index(
    documents,
    nlp,
    tokenizer,
    text_encoder,
    device,
    batch_size=8,
    remove_duplicate_token=True,
    discard_no_word=False,
):
    records = []
    term_to_indices = defaultdict(list)
    phrase_occurrences = 0
    token_occurrences = 0

    for batch_start in range(0, len(documents), batch_size):
        batch_documents = documents[batch_start : batch_start + batch_size]
        cleaned_texts = [clean_text(document["text"]) for document in batch_documents]
        batch_spans = [
            extract_important_spans(
                text,
                nlp,
                min_tokens=2,
                remove_duplicate=remove_duplicate_token,
                discard_no_word=discard_no_word,
            )
            for text in cleaned_texts
        ]

        token_embeddings_batch, offsets_batch = encode_chunk_batch(
            cleaned_texts,
            text_encoder=text_encoder,
            tokenizer=tokenizer,
            device=device,
        )

        for local_idx, document in enumerate(batch_documents):
            phrases, tokens = batch_spans[local_idx]
            token_embeddings = token_embeddings_batch[local_idx]
            offsets = offsets_batch[local_idx]
            document_idx = batch_start + local_idx

            phrase_records = collect_span_embedding_records(
                token_embeddings=token_embeddings,
                offsets=offsets,
                span_list=phrases,
                kind="phrase",
                document_idx=document_idx,
                title=document["title"],
                text=cleaned_texts[local_idx],
            )
            token_records = collect_span_embedding_records(
                token_embeddings=token_embeddings,
                offsets=offsets,
                span_list=tokens,
                kind="token",
                document_idx=document_idx,
                title=document["title"],
                text=cleaned_texts[local_idx],
            )

            for record in phrase_records + token_records:
                record["record_index"] = len(records)
                records.append(record)
                term_to_indices[(record["kind"], record["lookup_term"])].append(record["record_index"])

            phrase_occurrences += len(phrase_records)
            token_occurrences += len(token_records)

    return {
        "records": records,
        "term_to_indices": dict(term_to_indices),
        "metadata": {
            "file_path": file_path,
            "num_samples": num_samples,
            "text_encoder_name_or_path": text_encoder_name_or_path,
            "spacy_model_name": spacy_model_name,
            "remove_duplicate_token": remove_duplicate_token,
            "discard_no_word": discard_no_word,
            "num_phrase_occurrences": phrase_occurrences,
            "num_token_occurrences": token_occurrences,
            "num_total_occurrences": phrase_occurrences + token_occurrences,
            "num_unique_lookup_keys": len(term_to_indices),
        },
    }


def load_or_build_hotpot_embedding_index(output_path: Path = output_path):
    if output_path.exists():
        with open(output_path, "rb") as handle:
            embedding_store = pickle.load(handle)
        print(f"Loaded cached token db from {output_path}")
        return embedding_store

    documents, samples = build_hotpot_retrieval_dataset(file_path, num_samples=num_samples)
    print(f"documents={len(documents)} | samples={len(samples)}")

    nlp = ensure_nlp_loaded()
    tokenizer, text_encoder = ensure_text_encoder_loaded()
    embedding_store = build_phrase_token_embedding_index(
        documents=documents,
        nlp=nlp,
        tokenizer=tokenizer,
        text_encoder=text_encoder,
        device=device,
        batch_size=batch_size,
        remove_duplicate_token=remove_duplicate_token,
        discard_no_word=discard_no_word,
    )

    with open(output_path, "wb") as handle:
        pickle.dump(embedding_store, handle)

    print(f"Saved token db to {output_path}")
    return embedding_store


def lookup_hotpot_records(embedding_store, term: str, kind: str = "token", limit: int = 10):
    normalized_term = normalize_text(term)
    record_indices = embedding_store["term_to_indices"].get((kind, normalized_term), [])
    return [embedding_store["records"][idx] for idx in record_indices[:limit]]


embedding_store = load_or_build_hotpot_embedding_index(output_path=output_path)
embedding_store["metadata"]


In [ ]:
WORD_RE = re.compile(r"[A-Za-z][A-Za-z0-9_\-]*")


def merge_weights(custom_weights: Optional[dict[str, float]] = None) -> dict[str, float]:
    merged = dict(default_weights)
    if custom_weights:
        merged.update(custom_weights)
    total = sum(merged.values())
    if total <= 0:
        raise ValueError("Scoring weights must sum to a positive number.")
    return {name: value / total for name, value in merged.items()}


def tokenize_for_overlap(text: str) -> set[str]:
    return {
        token.lower()
        for token in WORD_RE.findall(text)
        if len(token) > 2
    }


def lexical_overlap_score(context_text: str, sense_profile: str) -> float:
    context_terms = tokenize_for_overlap(context_text)
    profile_terms = tokenize_for_overlap(sense_profile)
    if not context_terms or not profile_terms:
        return 0.0
    intersection = context_terms & profile_terms
    union = context_terms | profile_terms
    return float(len(intersection) / max(1, len(union)))


def resolve_target_span(
    context_text: str,
    target_text: str,
    target_char_span: Optional[tuple[int, int]] = None,
) -> tuple[tuple[int, int], list[str]]:
    warnings = []

    if target_char_span is not None:
        start_char, end_char = target_char_span
        if context_text[start_char:end_char] == target_text:
            return (start_char, end_char), warnings
        warnings.append(
            "Provided target_char_span does not exactly match target_text; attempting text-based fallback."
        )

    exact_matches = list(re.finditer(re.escape(target_text), context_text))
    if exact_matches:
        if len(exact_matches) > 1:
            warnings.append(
                f"Found {len(exact_matches)} exact matches for '{target_text}'. Using the first one."
            )
        match = exact_matches[0]
        return match.span(), warnings

    boundary_pattern = re.compile(rf"(?<!\w){re.escape(target_text)}(?!\w)", re.IGNORECASE)
    boundary_matches = list(boundary_pattern.finditer(context_text))
    if boundary_matches:
        warnings.append(
            "Exact case-sensitive match failed; using the first case-insensitive boundary match."
        )
        match = boundary_matches[0]
        return match.span(), warnings

    normalized_target = " ".join(target_text.split())
    normalized_pattern = re.compile(re.escape(normalized_target), re.IGNORECASE)
    normalized_matches = list(normalized_pattern.finditer(" ".join(context_text.split())))
    if normalized_matches:
        warnings.append(
            "Target matching required whitespace normalization. Token alignment may be imperfect."
        )

    raise ValueError(
        f"Could not locate target_text='{target_text}' inside the provided context_text."
    )


def mark_target_in_context(
    context_text: str,
    start_char: int,
    end_char: int,
    marker: str = "[TGT]",
) -> tuple[str, tuple[int, int]]:
    left_marker = f"{marker} "
    right_marker = f" {marker}"
    marked_text = (
        context_text[:start_char]
        + left_marker
        + context_text[start_char:end_char]
        + right_marker
        + context_text[end_char:]
    )
    marked_span = (start_char + len(left_marker), start_char + len(left_marker) + (end_char - start_char))
    return marked_text, marked_span


def build_sense_profile(candidate_sense: dict[str, Any]) -> str:
    lemma = candidate_sense.get("lemma", "").strip() or "(unknown lemma)"
    synonyms = [item.strip() for item in candidate_sense.get("synonyms", []) if str(item).strip()]
    definition = str(candidate_sense.get("definition", "")).strip() or "(no definition)"
    examples = [item.strip() for item in candidate_sense.get("examples", []) if str(item).strip()]

    lines = [
        f"Lemma: {lemma}",
        f"Sense ID: {candidate_sense.get('sense_id', '(missing sense id)')}",
        f"Synonyms: {', '.join(synonyms) if synonyms else '(none)'}",
        f"Definition: {definition}",
    ]
    if examples:
        lines.append("Examples: " + " | ".join(examples))
    return "\n".join(lines)


def candidate_senses_from_wordnet(
    target_text: str,
    pos: Optional[str] = "noun",
    exact_match_only: bool = True,
    include_related_info: bool = False,
    limit: Optional[int] = None,
) -> list[dict[str, Any]]:
    explorer = ensure_wordnet_explorer_loaded()
    details = explorer.get_synset_details(
        target_text,
        pos=pos,
        noun_only_words=(pos == "noun"),
        exact_match_only=exact_match_only,
    )

    candidates = []
    for item in details:
        candidates.append(
            {
                "sense_id": item["synset_name"],
                "lemma": target_text,
                "synonyms": list(item["lemmas"]),
                "definition": item["definition"],
                "examples": list(item["examples"]),
                "sense_profile": explorer._collect_synset_summary(
                    item,
                    include_related_info=include_related_info,
                ),
            }
        )

    if limit is not None:
        candidates = candidates[:limit]
    return candidates


def mean_pool(hidden_state: torch.Tensor, token_indices: list[int]) -> Optional[torch.Tensor]:
    if not token_indices:
        return None
    return hidden_state[token_indices].mean(dim=0)


def cosine_or_zero(left: Optional[torch.Tensor], right: Optional[torch.Tensor]) -> float:
    if left is None or right is None:
        return 0.0
    return float(F.cosine_similarity(left.unsqueeze(0), right.unsqueeze(0), dim=-1).item())


def collect_sequence_token_indices(
    sequence_ids: list[Optional[int]],
    offsets: list[tuple[int, int]],
    sequence_id: int,
    span: Optional[tuple[int, int]] = None,
) -> list[int]:
    token_indices = []
    start_char = None
    end_char = None
    if span is not None:
        start_char, end_char = span

    for token_idx, (current_sequence_id, (token_start, token_end)) in enumerate(zip(sequence_ids, offsets)):
        if current_sequence_id != sequence_id:
            continue
        if token_start == token_end:
            continue

        if span is None:
            token_indices.append(token_idx)
            continue

        if not (token_end <= start_char or token_start >= end_char):
            token_indices.append(token_idx)

    return token_indices


def get_context_window_indices(
    sequence_ids: list[Optional[int]],
    target_token_indices: list[int],
    window_size: int = 3,
) -> list[int]:
    if not target_token_indices:
        return []

    context_indices = [index for index, sequence_id in enumerate(sequence_ids) if sequence_id == 0]
    positions = {token_index: position for position, token_index in enumerate(context_indices)}

    start_position = positions[target_token_indices[0]]
    end_position = positions[target_token_indices[-1]]
    left = max(0, start_position - window_size)
    right = min(len(context_indices), end_position + window_size + 1)
    return context_indices[left:right]


def cross_attention_alignment_score(
    target_hidden: torch.Tensor,
    profile_hidden: torch.Tensor,
) -> tuple[float, Optional[torch.Tensor]]:
    if target_hidden.numel() == 0 or profile_hidden.numel() == 0:
        return 0.0, None

    query = F.normalize(target_hidden, p=2, dim=-1)
    key = F.normalize(profile_hidden, p=2, dim=-1)
    attention_logits = (query @ key.T) / math.sqrt(float(query.shape[-1]))
    attention = torch.softmax(attention_logits, dim=-1)
    attended_profile = attention @ profile_hidden
    aligned_cosine = F.cosine_similarity(target_hidden, attended_profile, dim=-1)
    return float(aligned_cosine.mean().item()), attention.detach().cpu()


def encode_joint_input(
    context_with_markers: str,
    sense_profile: str,
    max_length: int = 512,
) -> dict[str, Any]:
    tokenizer, text_encoder = ensure_text_encoder_loaded()
    encoded = tokenizer(
        context_with_markers,
        sense_profile,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation="longest_first",
        max_length=max_length,
    )
    sequence_ids = encoded.sequence_ids(0)

    model_inputs = {
        key: value.to(device)
        for key, value in encoded.items()
        if key != "offset_mapping"
    }
    outputs = text_encoder(**model_inputs, output_hidden_states=True)
    hidden_state = outputs.hidden_states[-2][0].detach().cpu()

    return {
        "encoding": encoded,
        "tokens": tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]),
        "sequence_ids": sequence_ids,
        "offsets": [tuple(map(int, item)) for item in encoded["offset_mapping"][0].tolist()],
        "hidden_state": hidden_state,
    }


def score_single_candidate(
    context_text: str,
    target_text: str,
    candidate_sense: dict[str, Any],
    target_char_span: Optional[tuple[int, int]] = None,
    weights: Optional[dict[str, float]] = None,
    window_size: int = 3,
    max_length: int = 512,
    verbose_warnings: bool = True,
) -> dict[str, Any]:
    merged_weights = merge_weights(weights)
    resolved_span, warnings = resolve_target_span(
        context_text=context_text,
        target_text=target_text,
        target_char_span=target_char_span,
    )
    marked_context, marked_span = mark_target_in_context(
        context_text,
        resolved_span[0],
        resolved_span[1],
    )
    sense_profile = candidate_sense.get("sense_profile") or build_sense_profile(candidate_sense)
    encoded = encode_joint_input(marked_context, sense_profile, max_length=max_length)

    target_token_indices = collect_sequence_token_indices(
        encoded["sequence_ids"],
        encoded["offsets"],
        sequence_id=0,
        span=marked_span,
    )
    if not target_token_indices:
        warnings.append(
            "Target token span could not be recovered from tokenizer offsets. "
            "The target may have been truncated or matching may be imperfect."
        )

    profile_token_indices = collect_sequence_token_indices(
        encoded["sequence_ids"],
        encoded["offsets"],
        sequence_id=1,
        span=None,
    )
    context_window_indices = get_context_window_indices(
        encoded["sequence_ids"],
        target_token_indices=target_token_indices,
        window_size=window_size,
    )

    hidden_state = encoded["hidden_state"]
    v_target = mean_pool(hidden_state, target_token_indices)
    v_context_window = mean_pool(hidden_state, context_window_indices)
    v_definition = mean_pool(hidden_state, profile_token_indices)

    cos_target_def = cosine_or_zero(v_target, v_definition)
    cos_window_def = cosine_or_zero(v_context_window, v_definition)

    if target_token_indices and profile_token_indices:
        cross_alignment, attention = cross_attention_alignment_score(
            hidden_state[target_token_indices],
            hidden_state[profile_token_indices],
        )
    else:
        cross_alignment, attention = 0.0, None

    lexical_overlap = lexical_overlap_score(context_text, sense_profile)

    final_score = (
        merged_weights.get("cos_target_def", 0.0) * cos_target_def
        + merged_weights.get("cos_window_def", 0.0) * cos_window_def
        + merged_weights.get("cross_alignment", 0.0) * cross_alignment
        + merged_weights.get("lexical_overlap", 0.0) * lexical_overlap
    )

    if warnings and verbose_warnings:
        print("\n".join(f"[warning] {message}" for message in warnings))

    return {
        "sense_id": candidate_sense.get("sense_id", "(missing sense id)"),
        "lemma": candidate_sense.get("lemma", target_text),
        "final_score": float(final_score),
        "cos_target_def": float(cos_target_def),
        "cos_window_def": float(cos_window_def),
        "cross_alignment": float(cross_alignment),
        "lexical_overlap": float(lexical_overlap),
        "sense_profile": sense_profile,
        "target_span": resolved_span,
        "marked_target_span": marked_span,
        "target_token_span": list(target_token_indices),
        "context_window_token_span": list(context_window_indices),
        "profile_token_span": list(profile_token_indices),
        "warnings": warnings,
        "debug": {
            "context_text": context_text,
            "marked_context": marked_context,
            "sense_profile": sense_profile,
            "tokens": encoded["tokens"],
            "offsets": encoded["offsets"],
            "sequence_ids": encoded["sequence_ids"],
            "attention": attention,
        },
    }


def score_candidate_senses(
    context_text: str,
    target_text: str,
    candidate_senses: list[dict[str, Any]],
    target_char_span: Optional[tuple[int, int]] = None,
    weights: Optional[dict[str, float]] = None,
    window_size: int = 3,
    max_length: int = 512,
    verbose_warnings: bool = True,
) -> list[dict[str, Any]]:
    if not candidate_senses:
        return []

    scored = [
        score_single_candidate(
            context_text=context_text,
            target_text=target_text,
            candidate_sense=candidate_sense,
            target_char_span=target_char_span,
            weights=weights,
            window_size=window_size,
            max_length=max_length,
            verbose_warnings=verbose_warnings,
        )
        for candidate_sense in candidate_senses
    ]
    scored.sort(key=lambda item: item["final_score"], reverse=True)
    return scored


def print_candidate_debug(
    context_text: str,
    target_text: str,
    candidate_sense: dict[str, Any],
    target_char_span: Optional[tuple[int, int]] = None,
    window_size: int = 3,
    max_length: int = 512,
) -> dict[str, Any]:
    result = score_single_candidate(
        context_text=context_text,
        target_text=target_text,
        candidate_sense=candidate_sense,
        target_char_span=target_char_span,
        window_size=window_size,
        max_length=max_length,
        verbose_warnings=True,
    )

    debug = result["debug"]
    print("Marked context:")
    print(debug["marked_context"])
    print()
    print("Sense profile:")
    print(debug["sense_profile"])
    print()
    print(f"Target char span: {result['target_span']}")
    print(f"Target token span: {result['target_token_span']}")
    print(f"Profile token span: {result['profile_token_span']}")
    print(f"Context window token span: {result['context_window_token_span']}")
    print()
    print("Tokenized joint input:")
    for idx, (token, sequence_id, offset) in enumerate(
        zip(debug["tokens"], debug["sequence_ids"], debug["offsets"])
    ):
        print(f"{idx:>3} | seq={sequence_id} | offset={offset} | token={token}")
    print()
    print("Intermediate scores:")
    print(f"  cos_target_def   = {result['cos_target_def']:.4f}")
    print(f"  cos_window_def   = {result['cos_window_def']:.4f}")
    print(f"  cross_alignment  = {result['cross_alignment']:.4f}")
    print(f"  lexical_overlap  = {result['lexical_overlap']:.4f}")
    print(f"  final_score      = {result['final_score']:.4f}")

    return result


def display_ranked_results(results: list[dict[str, Any]], keep_debug: bool = False) -> list[dict[str, Any]]:
    visible_results = []
    for item in results:
        visible = {
            "sense_id": item["sense_id"],
            "lemma": item["lemma"],
            "final_score": round(item["final_score"], 4),
            "cos_target_def": round(item["cos_target_def"], 4),
            "cos_window_def": round(item["cos_window_def"], 4),
            "cross_alignment": round(item["cross_alignment"], 4),
            "lexical_overlap": round(item["lexical_overlap"], 4),
            "sense_profile": item["sense_profile"],
            "warnings": item["warnings"],
        }
        if keep_debug:
            visible["debug"] = item["debug"]
        visible_results.append(visible)
    return visible_results


def extract_sentence_from_span(text: str, span: tuple[int, int]) -> str:
    start_char, end_char = span
    left_boundary = max(text.rfind(".", 0, start_char), text.rfind("?", 0, start_char), text.rfind("!", 0, start_char))
    right_candidates = [position for position in (text.find(".", end_char), text.find("?", end_char), text.find("!", end_char)) if position != -1]
    right_boundary = min(right_candidates) if right_candidates else len(text)
    sentence = text[left_boundary + 1 : right_boundary + 1].strip()
    return sentence if sentence else text.strip()


In [ ]:
director_context = "The board appointed a new director yesterday."
director_candidates = [
    {
        "sense_id": "director_manager",
        "lemma": "director",
        "synonyms": ["manager", "administrator", "executive"],
        "definition": "a person who controls or manages an organization or part of it",
        "examples": ["the director approved the budget"],
    },
    {
        "sense_id": "director_filmmaker",
        "lemma": "director",
        "synonyms": ["film director", "movie director"],
        "definition": "the person who supervises the actors and other staff while making a film",
        "examples": ["the director called for another take"],
    },
]

director_results = score_candidate_senses(
    context_text=director_context,
    target_text="director",
    candidate_senses=director_candidates,
    weights=default_weights,
    window_size=window_size,
    max_length=max_length,
)
display_ranked_results(director_results)


In [ ]:
career_context = "She had a long and successful career in science."
career_candidates = [
    {
        "sense_id": "career_profession",
        "lemma": "career",
        "synonyms": ["profession", "working life", "occupation"],
        "definition": "the general progression of a person's working life or professional achievements",
        "examples": ["he pursued a career in medicine"],
    },
    {
        "sense_id": "career_speed",
        "lemma": "career",
        "synonyms": ["rush", "dash", "bolt"],
        "definition": "the act of moving headlong at high speed",
        "examples": ["the horse set off in a sudden career"],
    },
]

career_results = score_candidate_senses(
    context_text=career_context,
    target_text="career",
    candidate_senses=career_candidates,
    weights=default_weights,
    window_size=window_size,
    max_length=max_length,
)
display_ranked_results(career_results)


In [ ]:
hotpot_director_records = lookup_hotpot_records(embedding_store, "director", kind="token", limit=5)
hotpot_director_records[:2]


In [ ]:
if hotpot_director_records:
    hotpot_record = hotpot_director_records[0]
    hotpot_context = extract_sentence_from_span(hotpot_record["text"], hotpot_record["span"])
    hotpot_candidates = candidate_senses_from_wordnet(
        "director",
        pos="noun",
        exact_match_only=True,
        include_related_info=False,
        limit=wordnet_candidate_limit,
    )
    hotpot_results = score_candidate_senses(
        context_text=hotpot_context,
        target_text="director",
        candidate_senses=hotpot_candidates,
        target_char_span=None,
        weights=default_weights,
        window_size=window_size,
        max_length=max_length,
    )
    print(f"title={hotpot_record['title']} | span={hotpot_record['span']}")
    print(hotpot_context)
    display_ranked_results(hotpot_results)
else:
    print("No 'director' token records were found in the current token db.")


In [ ]:
print_candidate_debug(
    context_text=director_context,
    target_text="director",
    candidate_sense=director_candidates[0],
    window_size=window_size,
    max_length=max_length,
)
